In [598]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme(style="whitegrid")


from sklearn.model_selection import(
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    cross_val_predict
)

from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report


RANDOM_SEED = 42

In [599]:
df = pd.read_csv('historical_data.csv')
df.columns

Index(['id', 'day', 'event_type', 'category', 'region', 'device',
       'account_age_days', 'num_prev_listings', 'prev_reports_30d',
       'verification_level', 'price', 'num_images', 'message_length',
       'contains_off_platform', 'urgency_words', 'payment_attempt',
       'time_to_first_response_min', 'is_suspicious'],
      dtype='str')

In [600]:
from math import log

num_cols = ['day',
            'account_age_days',
            'num_prev_listings',
            'prev_reports_30d',
            'verification_level',
            'price', 'num_images',
            'message_length',
            'contains_off_platform',
            'urgency_words',
            'payment_attempt',
            'time_to_first_response_min'
            ]
cat_cols = ['event_type', 'category', 'region', 'device']


drop_columns = [
    'event_type', 'category', 'region', 'device',"day", "id", 
    "is_suspicious", "payment_attempt", 'time_to_first_response_min',
    'price', 'num_prev_listings'
    ]

columns_used = [c for c in df.columns if c not in drop_columns]

X = df.drop(columns= drop_columns)
y = df['is_suspicious'].copy()


#X["fast_response_urgency"] = df["urgency_words"] / (df["time_to_first_response_min"] + 1)
#X["reports_per_listing"] = df["prev_reports_30d"] / (df["num_prev_listings"] + 1)
#X["urgency_density"] = df["urgency_words"] / (df["message_length"] + 1)
X["trust_score"] = df["verification_level"] * np.log(df["account_age_days"] + 1)
X["avg_listings_per_day"] = df["num_prev_listings"] / (df["account_age_days"] + 1)
#X["risk_score"] =((df["prev_reports_30d"] * 2) +(df["contains_off_platform"] * 3) +(df["payment_attempt"] * 3) +(df["urgency_words"]))
#X["high_verification"] = (df["verification_level"] >= 2).astype(int)
#X["prev_reports_30d_fraud_rate"] = df["prev_reports_30d"].map(
    #df.groupby("prev_reports_30d")["is_suspicious"].mean())
#X["off_platform_rate"] = df["contains_off_platform"].map(
#    df.groupby("contains_off_platform")["is_suspicious"].mean())



columns_used


['account_age_days',
 'prev_reports_30d',
 'verification_level',
 'num_images',
 'message_length',
 'contains_off_platform',
 'urgency_words']

In [601]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state= RANDOM_SEED,
    stratify=y
)

# Använd om inte K-fold eller cross validation
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state= RANDOM_SEED,
    stratify= y_train
)

X_tr.head()


,account_age_days,prev_reports_30d,verification_level,num_images,message_length,contains_off_platform,urgency_words,trust_score,avg_listings_per_day
1196,20.0,0,2,2,23,0,1,6.089045,0.190476
7713,24.1,1,1,0,69,1,0,3.222868,0.000000
8053,19.8,0,0,6,181,0,0,0.000000,0.240385
9826,130.3,0,0,4,208,0,0,0.000000,0.030465
11371,47.0,0,1,3,82,0,0,3.871201,0.041667


In [602]:
df.groupby("urgency_words")["is_suspicious"].mean()

urgency_words
0    0.093238
1    0.181665
Name: is_suspicious, dtype: float64

In [603]:


num_cols = [x for x in columns_used if x in num_cols]
cat_cols = [x for x in columns_used if x in cat_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])



preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

def make_model(model, k=16):
    return Pipeline([
        ("preprocess", preprocess),
        #("selector", SelectKBest(mutual_info_classif, k=k)),
        ("model", model)
    ])



In [604]:
lr = LogisticRegression(max_iter=200, class_weight="balanced")
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED,
        max_depth=5, min_samples_leaf=3, n_jobs=-1, class_weight="balanced")
dt = DecisionTreeClassifier(random_state=RANDOM_SEED)
gb = GradientBoostingClassifier(random_state=RANDOM_SEED)

In [605]:
model_lr = make_model(lr)
model_rf = make_model(rf)
model_dt = make_model(dt)
model_gb = make_model(gb)

In [606]:
model_lr.fit(X_tr, y_tr)
model_rf.fit(X_tr, y_tr)
model_dt.fit(X_tr, y_tr)
model_gb.fit(X_tr, y_tr)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers cont

In [607]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORING = "f1"

models = {
    "LogisticRegression": model_lr,
    "RandomForestClassifier": model_rf,
    "DecisionTree": model_dt,
    "GradientBoosting": model_gb
}


baseline_rows = []

for name, pipe in models.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring=SCORING)
    baseline_rows.append({"model": name, "mean": scores.mean(), "std": scores.std()})

baseline_table = pd.DataFrame(baseline_rows).sort_values("mean", ascending=False)
baseline_table

,model,mean,std
1,RandomForestClassifier,0.302294,0.013860
0,LogisticRegression,0.301701,0.008309
2,DecisionTree,0.194251,0.017663
3,GradientBoosting,0.124087,0.019604


In [608]:
predicted_y_log_reg = model_lr.predict(X_val)
predicted_y_rf = model_rf.predict(X_val)
predicted_y_dt = model_dt.predict(X_val)
predicted_y_gb= model_gb.predict(X_val)

print(classification_report(y_val, predicted_y_log_reg))
print(classification_report(y_val, predicted_y_rf))
print(classification_report(y_val, predicted_y_dt))
print(classification_report(y_val, predicted_y_gb))

              precision    recall  f1-score   support

           0       0.95      0.67      0.78      1724
           1       0.19      0.67      0.29       196

    accuracy                           0.67      1920
   macro avg       0.57      0.67      0.54      1920
weighted avg       0.87      0.67      0.73      1920

              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1724
           1       0.19      0.62      0.29       196

    accuracy                           0.69      1920
   macro avg       0.57      0.66      0.55      1920
weighted avg       0.87      0.69      0.75      1920

              precision    recall  f1-score   support

           0       0.91      0.90      0.90      1724
           1       0.18      0.20      0.19       196

    accuracy                           0.82      1920
   macro avg       0.55      0.55      0.55      1920
weighted avg       0.83      0.82      0.83      1920

              preci

In [609]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 3, 4],
    "model__class_weight": ["balanced"]
}

In [610]:
rfs = RandomForestClassifier(
    random_state=RANDOM_SEED,
    n_jobs=-1
)

pipe = make_model(rfs)

grid_rf = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,   # for example "f1"
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X_tr, y_tr)

pre_y_rf = grid_rf.predict(X_val)
print(classification_report(y_val, pre_y_rf))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1724
           1       0.19      0.62      0.29       196

    accuracy                           0.69      1920
   macro avg       0.57      0.66      0.54      1920
weighted avg       0.86      0.69      0.75      1920



In [611]:

pre_y_rf = grid_rf.predict(X_val)
print(classification_report(y_val, pre_y_rf))

grid_rf.best_params_

              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1724
           1       0.19      0.62      0.29       196

    accuracy                           0.69      1920
   macro avg       0.57      0.66      0.54      1920
weighted avg       0.86      0.69      0.75      1920



{'model__class_weight': 'balanced',
 'model__max_depth': 5,
 'model__min_samples_leaf': 3,
 'model__min_samples_split': 10,
 'model__n_estimators': 200}

In [612]:
lr = LogisticRegression(max_iter=200, class_weight="balanced")
model_l = make_model(lr)

# parameter grid
param_grid_lr = {
    "model__penalty": ["l1", "l2"],
    "model__C": [0.001, 0.01, 0.1, 1, 10],
    "model__solver": ["liblinear"],
    "model__class_weight": [None, "balanced"],
    "model__max_iter": [200]
}

# grid search
grid_lr = GridSearchCV(
    estimator=model_l,
    param_grid=param_grid_lr,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    verbose=2
)

# fit
grid_lr.fit(X_tr, y_tr)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\simon\Desktop\Marketplace\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\simon\Desktop\Marketplace\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...x_iter=200))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.001, 0.01, ...], 'model__class_weight': [None, 'balanced'], 'model__max_iter': [200], 'model__penalty': ['l1', 'l2'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the 

In [613]:
print("Best Parameters:", grid_lr.best_params_)
print("Best Score:", grid_lr.best_score_)

Best Parameters: {'model__C': 0.1, 'model__class_weight': 'balanced', 'model__max_iter': 200, 'model__penalty': 'l1', 'model__solver': 'liblinear'}
Best Score: 0.3004704098465489


In [614]:
pre_y_lr = grid_lr.predict(X_val)
print(classification_report(y_val, pre_y_lr))

              precision    recall  f1-score   support

           0       0.95      0.66      0.78      1724
           1       0.18      0.67      0.29       196

    accuracy                           0.66      1920
   macro avg       0.57      0.67      0.53      1920
weighted avg       0.87      0.66      0.73      1920



In [615]:
y_prob = model_lr.predict_proba(X_val)[:,1]

y_pred = (y_prob > 0.5).astype(int)

from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_val, y_pred))

[[1152  572]
 [  65  131]]
